# CSD and MSMT-CSD on a 60 degree crossing — the Python arm

The **CSD half of the manuscript simulation**, in Python, on a simulation
environment proven identical to `smi_manuscript_60deg.m`'s.

This notebook does not fit SMI. It builds the same ground truth, the same
kernel and the same noise-free signal the `.m` file builds, adds Rician noise,
deconvolves with **SSST-CSD and MSMT-CSD through the MRtrix3 binaries**, scores
the peaks, and **exports everything the `.m` file needs to draw the figures** —
including the noisy signal itself, so the SMI arm can be run on the very same
realisations.

| arm | what it is |
|---|---|
| **SSST-CSD** | `dwi2fod csd` on the top shell — MRtrix3 3.0.4, the binary |
| **MSMT-CSD** | `dwi2fod msmt_csd` on all four shells, 3 tissues — the binary |

Nothing here reimplements MRtrix. `dwi2fod`, `dwiextract`, `mrinfo` and
`sh2peaks` are called as subprocesses; the only MRtrix behaviour implemented
locally is reading and writing its image format, and `mrinfo` checks that on
every run.

### The healthy kernel, and what "equivalent to SMI's" means here

The tissue is SMI's healthy kernel, `[f Da Depar Deperp fw] = [0.60 2.0 2.0
0.50 0.02]` — the same compartment dimensions the published Monte Carlo used.

The response handed to CSD is that kernel **convolved with the same Watson
`kappa = 16` the ground truth disperses each fibre population by**:

$$r_l(b) \;=\; K_l(b)\; p_l^{\mathrm{Watson}}(\kappa)\; \sqrt{(2l+1)\,4\pi}$$

That is the *dispersion-matched* response, and it is the honest single-fibre
response for this simulation: a "fibre" here is not a delta, it is a dispersed
population, and a response estimated from real white matter would have absorbed
that dispersion too. Step 3 prints it next to the delta response and next to the
three `dwi2response` estimators, so you can see where it lands.

### Identity with the `.m` file is measured, not asserted

Step 0 recomputes the protocol, the SH basis, the Watson mixture, the kernel
invariants and the noise-free signal, and compares each against arrays dumped
straight out of Octave by `dump_reference.m`. **The noise-free signal agrees to
~7e-15.** That is the claim that both halves simulate the same experiment.

**What is deliberately not identical: the noise realisations.** The `.m` file
draws from Octave's legacy generator and this notebook from numpy's PCG64, and
those streams cannot be made bit-identical without reimplementing one inside the
other. So the noisy signal is **exported** rather than assumed reproducible —
Step 8 writes it out, and the `.m` file can fit SMI to exactly these voxels if
common random numbers are wanted.

```
 Step 0   prove the environment matches the .m   -> max|err| per array
 Step 1   the acquisition protocol               -> a real HCP 3-shell scheme
 Step 2   the ground truth fibre geometry        -> a 60 degree crossing
 Step 3   the kernel, and the response CSD gets  -> dispersion matched
 Step 4   forward convolution                    -> noise-free signal
 Step 5   Rician noise, one block per SNR        -> the measured data
 Step 6   dwi2fod csd / msmt_csd                 -> the two arms
 Step 7   peaks, angular error, spurious         -> where does it stop working?
 Step 8   export for the .m file                 -> SH, scores and the signal
 Figures  1-3 (a preview; the .m draws the manuscript versions)
 Step 9   how to scale this up
```

In [ ]:
# ============================ Configuration ============================
# Every knob is here. Nothing below this cell needs editing to retune the
# experiment. The values mirror smi_manuscript_60deg.m's Configuration block;
# where a name matches, the meaning matches.
import os, sys, subprocess, time
import numpy as np

# Locate deconv_comparison/ by walking up from wherever the kernel started.
d = os.getcwd(); PKGDIR = ''
for _ in range(6):
    if os.path.exists(os.path.join(d, 'deconv_comparison', 'smi_sim.py')):
        PKGDIR = os.path.join(d, 'deconv_comparison'); break
    if os.path.exists(os.path.join(d, 'smi_sim.py')):
        PKGDIR = d; break
    dn = os.path.dirname(d)
    if dn == d: break
    d = dn
if not PKGDIR:
    raise RuntimeError('cannot find deconv_comparison/smi_sim.py from %s' % os.getcwd())
sys.path.insert(0, PKGDIR)
import smi_sim as S
print('package     : %s' % PKGDIR)

VERDICT = {True: 'ok', False: '** FAILED **'}
CHECKS = []
def CHECK(label, ok, detail=''):
    """Record and print one check. Reading only these lines is a full audit."""
    CHECKS.append((label, bool(ok)))
    print('   CHECK %-52s %s%s' % (label, VERDICT[bool(ok)],
                                   ('   ' + detail) if detail else ''))
    return bool(ok)

# -------------------------------------------------------------- the tissue
# SMI's compartment vector is [f Da Depar Deperp fw]. This is the HEALTHY
# kernel -- the same one the published Monte Carlo used, and the same one
# smi_manuscript_60deg.m runs as its first preset. The edema kernel is not
# simulated here.
PRESET = 'healthy'
K_WM   = [0.60, 2.0, 2.0, 0.50, 0.02]
D_FW   = 3.0        # free water diffusivity, um^2/ms
D_GM   = 0.8        # grey matter diffusivity, um^2/ms -- ONLY used to build the
                    # isotropic response MSMT-CSD needs as its second tissue.
                    # No simulated voxel contains grey matter.
KAPPA  = 16.0       # Watson concentration of each fibre population. Finite, not
                    # a delta -- and the reason the response below is dispersion
                    # matched rather than the exact delta kernel response.

# ------------------------------------------------------------- the geometry
ANGLES = [60]       # crossing angles, degrees. Measured on the noise-free truth,
                    # 60 separates at every Lmax and every KAPPA tested, where 30
                    # never separates at KAPPA = 16 and 45 only from Lmax 6 up.
AXIS1  = np.array([0.30, -0.50, 0.81])    # off every coordinate plane
AXIS1  = AXIS1 / np.linalg.norm(AXIS1)

# ------------------------------------------------------------ the experiment
SMOKE_TEST = True                 # <-- THE KNOB. False = manuscript settings.

if SMOKE_TEST:
    NREP      = 27                # realisations per condition PER SNR
    SNR_LIST  = [10, 30, np.inf]
    LMAX_LIST = [6]
else:
    NREP      = 1000
    SNR_LIST  = [5, 10, 20, 30, 50, 100, np.inf]
    LMAX_LIST = [4, 6, 8]
# NREP * len(ANGLES) must factor into three integers > 1: SMI.vectorize takes a
# different branch if any spatial dimension is a singleton, and the .m file will
# read these voxels back on the same grid. pick_grid errors loudly if it cannot.

LMAX_GT   = 8            # ground truth angular order. A CEILING, not a choice:
                         # SMI's kernel invariants K_l are undefined above l = 8.
CS_PHASE  = 0            # 0 == MRtrix's SH basis exactly. This is what lets an
                         # SMI fODF and an MRtrix FOD be scored by the same code.
PROTOCOL  = 'hcp_real_3shell.txt'
B0_SNAP   = 0.05         # ms/um^2. Any shell below this is treated as exactly
                         # b = 0. A real .bval records EFFECTIVE b, so its
                         # "b = 0" volumes are b = 5 s/mm^2; snapping restores
                         # the exact identity S(0)/S0 = 1.
NDIR_Q    = 3000         # quadrature directions for projecting a sampled fODF
NDIR_E    = 1500         # evaluation directions for the peak finder
SEED      = 31415        # RNG seed, offset per SNR block

# ------------------------------------------------- the response given to CSD
# 'dispersed' = the kernel convolved with the Watson at KAPPA, i.e. the response
#               of one dispersed fibre population -- what this simulation's
#               "single fibre" actually is.
# 'delta'     = the exact analytic kernel response, sharper than any real tissue.
# Both are built below and printed side by side; this selects which one the
# binaries are given.
RESPONSE_MODE = 'dispersed'

# ------------------------------------------------------------ the MRtrix arms
RUN_MRTRIX   = True      # False builds the signal and skips the deconvolution
RUN_SH2PEAKS = True      # Step 7's independent cross-check against sh2peaks
MDIR = os.path.join(PKGDIR, 'mrtrix')     # gitignored scratch for MRtrix images
EDIR = os.path.join(PKGDIR, 'export')     # where the .m file reads results from
TAG  = 'csdpy_' + PRESET

# ------------------------------------------------------------------ assembled
NSNR       = len(SNR_LIST)
NCOND      = len(ANGLES)
SIGMA_LIST = [0.0 if np.isinf(s) else 1.0 / s for s in SNR_LIST]
SNR_LABEL  = ['inf' if np.isinf(s) else '%g' % s for s in SNR_LIST]
SNR_ORD    = sorted(range(NSNR), key=lambda i: SNR_LIST[i])   # ascending, Inf last

AXES_GT = []
for a in ANGLES:
    AXES_GT.append([AXIS1] if a == 0 else [AXIS1, S.rotate_about(AXIS1, a)])
COLHDR = ['single' if a == 0 else '%d deg' % a for a in ANGLES]

print('\n=== SSST-CSD / MSMT-CSD manuscript simulation (Python arm) ===')
if SMOKE_TEST:
    print('*** SMOKE_TEST = True: reduced sweep, indicative numbers only ***')
print('kernel      : %-8s [f Da Depar Deperp fw] = %s, extra-axonal = %.2f'
      % (PRESET, K_WM, 1 - K_WM[0] - K_WM[4]))
print('condition   : %s deg crossing,  kappa = %g' % (ANGLES, KAPPA))
print('response    : %s' % RESPONSE_MODE)
print('protocol    : %s,  CS_phase %d,  truth at Lmax %d' % (PROTOCOL, CS_PHASE, LMAX_GT))
print('sweep       : SNR %s x Lmax %s, %d voxels per block'
      % (SNR_LABEL, LMAX_LIST, NCOND * NREP))
print('MRtrix arms : %s' % VERDICT[RUN_MRTRIX])

## Step 0 — prove this notebook simulates the same experiment as the `.m` file

The whole point of putting the CSD arms in Python is undermined if the signal
they see is not the signal the SMI arm sees. So before anything is simulated,
every shared piece of the forward model is recomputed here and compared against
arrays dumped straight out of Octave.

Run this once, from `deconv_comparison/`, to produce the Octave side:

```
octave-cli --no-gui -q dump_reference.m
```

The comparison covers the protocol reader, both direction sets, the SH basis in
**both** `CS_phase` conventions, the Watson mixture, the fODF projection, the
kernel invariants `K_l(b)`, and the noise-free signal by two independent routes.

Two rows are compared **relatively** rather than absolutely, and the reason is
worth knowing rather than discovering later:

- **the b values differ by 1 ulp.** Octave's `textscan` and Python's `float()`
  round decimal strings like `2.99` differently in the last bit — measured, 80
  of 288 values, 1.5e-16 relative. Both are reading the same text file.
- **the Watson amplitudes are `exp(kappa) ≈ 9e6`** at `kappa = 16`, so an
  absolute tolerance there is really a demand for 22 significant digits.

If `data/` has no reference, this step says so and the notebook continues — the
simulation still runs, it is just no longer *proven* to match.

In [ ]:
# The full comparison lives in check_python_vs_octave.py so it can also be run
# from a shell, and so this notebook cannot quietly use a weaker version of it.
ref_ok = os.path.exists(os.path.join(PKGDIR, 'data', 'ref_S_clean.shape'))
if not ref_ok:
    print('Step 0: SKIPPED -- data/ has no Octave reference.')
    print('        Run this once, from %s:' % PKGDIR)
    print('            octave-cli --no-gui -q dump_reference.m')
    print('        The simulation below still runs; it is just not proven to')
    print('        match smi_manuscript_60deg.m.')
    CHECK('Python and Octave simulate the same experiment', False, '(not run)')
else:
    r = subprocess.run([sys.executable, 'check_python_vs_octave.py'],
                       cwd=PKGDIR, capture_output=True, text=True)
    print(r.stdout.rstrip())
    if r.stderr.strip():
        print(r.stderr.rstrip())
    CHECK('Python and Octave simulate the same experiment', r.returncode == 0,
          'every array compared above')

## Step 1 — the acquisition protocol

A **real HCP 3-shell scheme**, tracked as text at
`protocol/hcp_real_3shell.txt`. 288 volumes: 18 at b = 0 plus 90 each at
nominal b = 1, 2, 3 ms/µm².

Two properties of real acquisitions show up immediately and neither is tidied
away, because both are things code that only ever saw synthetic protocols would
get wrong:

- **the b = 0 volumes are not b = 0 as acquired** — they are b = 5 s/mm², because
  a `.bval` records *effective* b and the imaging gradients contribute a little
  weighting of their own. `B0_SNAP` sets them to exactly 0.
- **the b values jitter within each shell** — 18 distinct values across the
  scheme. The forward model uses each volume's exact b; MRtrix does its own
  shell clustering, and Step 6 checks the two agree.

Expect a loud warning about the gradient directions. It is deliberate: the
supplied `.bvec` is unit only to `1.1e-6`, being written at seven significant
figures, and left uncorrected that degrades the zonal-response identity at
Lmax 8 from `1e-15` to `5e-7`.

In [ ]:
bvals, bvecs = S.load_protocol(PROTOCOL, b0_snap=B0_SNAP, verbose=True)
Ndwi = len(bvals)
b_shell, shell_id = S.group_shells(bvals)
dw = bvals > 0

print('Step 1: %d volumes, %d distinct b values' % (Ndwi, len(np.unique(bvals))))
print('   shells as binned here:')
n_shell = np.array([int((shell_id == k).sum()) for k in range(len(b_shell))])
for k, (b, n) in enumerate(zip(b_shell, n_shell)):
    print('        %d   b = %7.4f ms/um^2 (%6.1f s/mm^2)   %3d volumes'
          % (k, b, 1000 * b, n))

# CHECK. The real protocol is 18 b=0 plus three shells of 90. If the binning
# splits a shell, every per-shell quantity below is attached to the wrong b.
CHECK('shell sizes are [18 90 90 90]', list(n_shell) == [18, 90, 90, 90],
      'got %s' % list(n_shell))
CHECK('b = 0 shell is exactly zero', float(b_shell[0]) == 0.0,
      'B0_SNAP = %g' % B0_SNAP)
CHECK('gradient directions are unit', float(np.abs(np.linalg.norm(bvecs, axis=1) - 1).max()) < 1e-15)

## Step 2 — the ground truth fibre geometry

Two equal fibre populations crossing at 60 degrees. Each is a **Watson**
distribution with concentration `kappa = 16`, not a delta.

That choice is the one most likely to be questioned, so: a response function
estimated from real white matter has already absorbed fibre dispersion. A delta
ground truth would create a response/truth mismatch that does not exist in
practice and would flatter whichever method sharpens most. It is also what makes
the dispersion-matched response in Step 3 the *correct* response rather than a
handicap.

The fODF is sampled on `NDIR_Q` quadrature directions and projected onto `plm`
in SMI's normalised convention `p_00 = 1`, in which **the fODF integrates to 1
in every voxel**.

One number below looks alarming and is a finding rather than a failure: **the
band-limited ground truth is negative over roughly 40% of the sphere.**
Truncating a Watson mixture rings, and the rings cross zero. So the
non-negativity constraint CSD imposes is a regularizer, not a statement of fact
— the truth does not satisfy it either.

In [ ]:
dq = S.uniform_sphere_dirs(NDIR_Q)
L_gt = S.sh_degrees(LMAX_GT)

fodf_gt = np.zeros((NDIR_Q, NCOND))
plm_gt  = np.zeros((NCOND, S.n_coef(LMAX_GT) - 1))
sh_gt   = np.zeros((NCOND, S.n_coef(LMAX_GT)))
sep_deg = np.zeros(NCOND)

print('Step 2: %d condition(s)' % NCOND)
for ic in range(NCOND):
    f = np.zeros(NDIR_Q)
    for ax in AXES_GT[ic]:
        f += S.watson_amp(dq, ax, KAPPA)
    fodf_gt[:, ic] = f
    plm_gt[ic] = S.fodf_to_plm(f, dq, LMAX_GT, CS_PHASE)
    sh_gt[ic]  = S.plm_to_sh(plm_gt[ic], LMAX_GT)
    if len(AXES_GT[ic]) == 2:
        sep_deg[ic] = np.degrees(np.arccos(abs(AXES_GT[ic][0] @ AXES_GT[ic][1])))
    print('   condition %d: %2d deg, %d population(s), measured separation %.6f deg'
          % (ic, ANGLES[ic], len(AXES_GT[ic]), sep_deg[ic]))

# CHECK 1. The angle between the axes must be the angle asked for. A wrong
# rotation axis would otherwise look plausible all the way to the end.
CHECK('crossing angle is what was requested',
      float(np.abs(sep_deg - np.array(ANGLES, dtype=float)).max()) < 1e-9)

# CHECK 2. Reconstruct each fODF from its plm and integrate over the sphere. In
# the p_00 = 1 convention the integral is 1 by construction, so anything else
# means the convention has been broken.
Yq   = S.get_even_SH(dq, LMAX_GT, CS_PHASE)
amp  = sh_gt @ Yq.T
mass = amp.mean(axis=1) * 4 * np.pi
CHECK('fODF integrates to 1', float(np.abs(mass - 1).max()) < 1e-3,
      'max|int-1| = %.2e' % float(np.abs(mass - 1).max()))

# CHECK 3. The fODF as SAMPLED is a sum of Watson distributions, positive by
# construction. (Unnormalised, so the floor is 1, not 0.)
CHECK('sampled fODF >= 0', float(fodf_gt.min()) >= 0, 'min = %+.4f' % fodf_gt.min())

print('   band-limited truth at Lmax %d -- peak, minimum, %% of sphere below 0:' % LMAX_GT)
for ic in range(NCOND):
    print('     %2d deg : peak %+.4f, min %+.4f, negative over %4.1f%% of directions'
          % (ANGLES[ic], amp[ic].max(), amp[ic].min(), 100 * (amp[ic] < 0).mean()))
print('   isotropic floor 1/(4*pi) = %.4f  (MRtrix iFOD2 default cutoff is 0.05)'
      % (1 / (4 * np.pi)))

## Step 3 — the kernel, and the response CSD is actually given

SMI has no response function. It has a **kernel**: the Standard Model
compartment vector `[f Da Depar Deperp fw]`, whose rotational invariants
`K_l(b)` are what the forward model convolves with. The conversion to the object
MRtrix stores in a response `.txt` is one line —

$$r_l(b) = K_l(b)\,\sqrt{(2l+1)\,4\pi}$$

— and that is the **delta** response: the response of a single fibre with no
dispersion at all.

**But no fibre in this simulation is a delta.** Each population is a Watson at
`kappa = 16`. The response of *one such population* is the kernel convolved with
that Watson, which in zonal harmonics is a product:

$$r_l(b) = K_l(b)\; p_l^{\mathrm{Watson}}(\kappa)\; \sqrt{(2l+1)\,4\pi}$$

with $p_l^{\mathrm{Watson}}$ the Watson's own normalised zonal coefficients
($p_0 = 1$). That is what `RESPONSE_MODE = 'dispersed'` builds, and it is the
response a perfect `dwi2response` run on this data would recover.

The table below puts it next to the delta response and next to the three real
`dwi2response` estimators measured on the phantom
(`Reports/REPORT_SMI_deconvolution_MonteCarlo.md`, section 6.3). The
dispersion-matched response should land **between** them — blunter than the
delta, close to what estimation actually produces.

**What changes because of this choice.** Deconvolving with the delta response
asks CSD to reproduce the Watson lobes; deconvolving with the dispersed response
asks it to reproduce two near-deltas. Peak *orientations* are unaffected — which
is what Step 7 scores — but the fODFs will look sharper than the SMI arm's, and
that is a convention difference rather than a result.

In [ ]:
K = K_WM
Kmat = S.Kell_matrix(K, b_shell, LMAX_GT, D_FW)

# The Watson zonal coefficients, two independent ways: the sampled-grid
# projection the ground truth itself uses, and a 1-D quadrature with no
# spherical harmonics anywhere.
pl_watson = S.watson_zonal_pl(KAPPA, LMAX_GT, dirs_q=dq, CS_phase=CS_PHASE)
pl_exact  = S.watson_zonal_pl_exact(KAPPA, LMAX_GT)

print('Step 3: kernel %s, D_FW = %g' % (K, D_FW))
print('   Watson p_l at kappa = %g (l = 0,2,..,%d):' % (KAPPA, LMAX_GT))
print('        SH projection : %s' % np.array2string(pl_watson, precision=5))
print('        1-D quadrature: %s' % np.array2string(pl_exact, precision=5))

# CHECK. Two routes to the same dispersion factors. They are not identical --
# the SH route is band limited at LMAX_GT and the quadrature is not -- so the
# residual is the band-limiting error, not a mistake.
CHECK('Watson p_l agrees by two independent routes',
      float(np.abs(pl_watson - pl_exact).max()) < 2e-4,
      'max|err| = %.2e, band limiting' % float(np.abs(pl_watson - pl_exact).max()))

R_delta = S.response_zh(K, b_shell, LMAX_GT, D_FW)
R_disp  = S.response_zh(K, b_shell, LMAX_GT, D_FW, pl=pl_watson)
RESP = {'delta': R_delta, 'dispersed': R_disp}[RESPONSE_MODE]

print('\n   the response, normalised to l = 0, at the top shell b = %.2f:' % b_shell[-1])
print('        %-26s %s' % ('delta (exact kernel)',
                            np.array2string(R_delta[-1] / R_delta[-1, 0], precision=4)))
print('        %-26s %s' % ('dispersion matched',
                            np.array2string(R_disp[-1] / R_disp[-1, 0], precision=4)))
for nm, fn in [('dwi2response dhollander', 'resp_wm.txt'),
               ('dwi2response tournier',   'resp_wm_tournier.txt'),
               ('dwi2response fa',         'resp_wm_fa.txt')]:
    p = os.path.join(PKGDIR, 'mrtrix_responses', fn)
    if os.path.exists(p):
        Re = S.read_response(p)
        print('        %-26s %s' % (nm, np.array2string(Re[-1] / Re[-1, 0], precision=4)))
print('        (the estimators were measured on the SUPERSEDED synthetic protocol,')
print('         Reports/REPORT_SMI_deconvolution_MonteCarlo.md section 6.3, so they')
print('         are here for scale, not as a like-for-like comparison)')

# NOT a check, a measurement: how much of the gap between the delta response and
# a real estimated one is explained by fibre dispersion alone.
r2_delta = R_delta[-1, 1] / R_delta[-1, 0]
r2_disp  = R_disp[-1, 1] / R_disp[-1, 0]
print('\n   r_2/r_0 at b = %.2f: delta %+.4f -> dispersed %+.4f  (%.0f%% blunter)'
      % (b_shell[-1], r2_delta, r2_disp, 100 * (1 - r2_disp / r2_delta)))
print('   Using RESPONSE_MODE = %r for both MRtrix arms.' % RESPONSE_MODE)

## Step 4 — forward convolution to a noise-free signal

The signal is the fODF convolved with the kernel. In spherical harmonics
convolution is a product:

$$S(u)/S_0 = \sum_{lm} K_l(b)\, p_{lm}\, Y_{lm}(u)\, \sqrt{(2l+1)\,4\pi}$$

To make sure the harmonic machinery is right, the same signal is also computed
with **no spherical harmonics anywhere**, by direct numerical convolution over
the quadrature grid. Those two do *not* agree to machine precision and should
not: the harmonic route is band limited at `LMAX_GT` and the direct sum is not.
The residual **is** the ground truth's band-limiting error, and Step 5 puts it
next to the noise so you can see which dominates.

In [ ]:
S_clean = np.zeros((NCOND, Ndwi))
S_direct = np.zeros((NCOND, Ndwi))
for ic in range(NCOND):
    S_clean[ic]  = S.forward_signal(plm_gt[ic], K, bvals, bvecs, LMAX_GT, CS_PHASE, D_FW)
    S_direct[ic] = S.forward_signal_direct(fodf_gt[:, ic], dq, K, bvals, bvecs, D_FW)

print('Step 4: noise-free signal')
print('   mean signal per shell:')
print('        b      ' + ''.join('%10s' % c for c in COLHDR))
for k in range(len(b_shell)):
    print('     %5.2f   ' % b_shell[k]
          + ''.join('%10.4f' % S_clean[ic, shell_id == k].mean() for ic in range(NCOND)))

# CHECK 1. The b = 0 signal must be exactly 1, because sigma is set to 1/SNR
# below and that only means the requested SNR if S0 = 1.
e_s0 = float(np.abs(S_clean[:, ~dw].mean(axis=1) - 1).max())
CHECK('S(b=0) == 1 exactly', e_s0 < 1e-12, 'max|S-1| = %.2e' % e_s0)

# CHECK 2. Harmonics against direct convolution. The residual is the truth's
# band-limiting error, not an error in the code.
e_sh = float(np.abs(S_clean - S_direct).max())
print('   harmonics vs direct convolution: max|err| = %.2e' % e_sh)
print('         (band limiting, not an error -- it falls as LMAX_GT rises)')

## Step 5 — Rician noise, one block of voxels per SNR

Complex Gaussian noise is added to a real signal and the magnitude taken, which
is exactly Rician:

$$S_{\text{noisy}} = \sqrt{(S + \sigma n_1)^2 + (\sigma n_2)^2}, \qquad n_1, n_2 \sim N(0,1)$$

with `sigma = 1/SNR` against `S0 = 1`.

The sweep is laid out as **one contiguous block of voxels per SNR**, each block
holding all conditions at `NREP` realisations, in the same voxel order the `.m`
file uses — so an exported voxel index means the same thing on both sides.

**This is the one step that is not bit-identical to the `.m` file**, and it
cannot be: Octave's `randn('seed',...)` legacy generator and numpy's PCG64 are
different streams. What is guaranteed instead is that the noise *process* is the
same, verified below by recovering sigma from the data rather than trusting the
value typed in — and that the realisations themselves are **exported** in Step 8,
so the SMI arm can be fitted to exactly these voxels.

In [ ]:
NVOX_SNR = NCOND * NREP
NVOX     = NVOX_SNR * NSNR
GRID_SNR = S.pick_grid(NVOX_SNR)
GRID_ALL = S.pick_grid(NVOX)

cond_id = np.tile(np.repeat(np.arange(NCOND), NREP), NSNR)
snr_id  = np.repeat(np.arange(NSNR), NVOX_SNR)
S_rep   = S_clean[cond_id, :]
S_noisy = np.zeros((NVOX, Ndwi))

for isnr in range(NSNR):
    rows = np.where(snr_id == isnr)[0]
    rng  = np.random.default_rng(SEED + isnr)     # per-SNR seed, as the .m does
    S_noisy[rows] = S.rician(S_rep[rows], SIGMA_LIST[isnr], rng)

print('Step 5: %d condition(s) x %d reps x %d SNR = %d voxels'
      % (NCOND, NREP, NSNR, NVOX))
print('   one block per SNR: %d voxels, grid %s;  whole sweep grid %s'
      % (NVOX_SNR, GRID_SNR, GRID_ALL))

# CHECK 1. Recover sigma from the simulated data at every SNR, rather than
# trusting the value typed in. At SNR = inf this is the statement that the
# "noisy" signal is bit identical to the noise-free one.
for isnr in range(NSNR):
    rows = np.where(snr_id == isnr)[0]
    rblk = S_noisy[rows] - S_rep[rows]
    sg   = SIGMA_LIST[isnr]
    if sg == 0:
        CHECK('SNR %-4s noise free, residual exactly zero' % SNR_LABEL[isnr],
              float(np.abs(rblk).max()) == 0.0)
    else:
        s_hat = float(rblk.std())
        rel = abs(s_hat - sg) / sg
        CHECK('SNR %-4s recovered sigma %.5f vs %.5f' % (SNR_LABEL[isnr], s_hat, sg),
              rel < 0.10, '%.1f%% off' % (100 * rel))

# CHECK 2. A magnitude is strictly positive.
CHECK('signal strictly positive', float(S_noisy.min()) > 0,
      'min = %.4f' % S_noisy.min())

sg_nz = [s for s in SIGMA_LIST if s > 0]
if sg_nz:
    print('   smallest non-zero sigma %.4f is %.0fx the band-limiting error %.2e,'
          % (min(sg_nz), min(sg_nz) / e_sh, e_sh))
    print('   so noise dominates the band limit at every finite SNR in the sweep')

## Step 6 — the MRtrix arms: `dwi2fod csd` and `dwi2fod msmt_csd`

The signal Step 5 built is written out as an MRtrix image with an embedded
`dw_scheme`, the binaries are invoked, and their fODFs are read back. Four
things have to be right and each gets a check rather than an assumption.

**1. MRtrix must bin the shells the way this notebook did.** The b values jitter
and MRtrix does its own clustering. The shells are read back with
`mrinfo -shell_bvalues`, checked, and **the response is evaluated at MRtrix's own
average b per shell**, not at the nominal 0/1/2/3 — otherwise every response row
is attached to a b nobody acquired at.

**2. The response has to survive the round trip.** It is written, read back off
disk, and compared against the array that was written.

**3. Lmax has to match.** `dwi2fod` gets `-lmax` matching `LMAX_LIST` entry by
entry, and one response file is written per Lmax with exactly the right number
of columns, so there is no silent truncation and Step 7's ceiling is the correct
bound.

**4. Scale is not comparable, and does not need to be.** An SMI fODF has
`p_00 = 1` and integrates to 1; an MRtrix FOD is unnormalised and its amplitude
carries apparent fibre density. Step 7's peak finder subtracts **each voxel's
own l = 0 term** rather than the constant `1/(4π)`, which makes it scale-free.

**MSMT-CSD needs three tissues and this simulation contains one.** Its second and
third responses are idealized isotropic ones at `D_GM` and `D_FW`. No simulated
voxel contains grey matter or free-standing CSF, so what MSMT assigns to those
compartments is a measurement of its own leakage — printed below rather than
discarded.

In [ ]:
ARMS = []          # each: dict(name=..., sh={iL: [NVOX x ncoef]})

if not RUN_MRTRIX:
    print('Step 6: SKIPPED (RUN_MRTRIX = False)')
else:
    os.makedirs(MDIR, exist_ok=True)
    print('Step 6: the MRtrix arms, in %s' % MDIR)

    def run(cmd, what):
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            print(r.stderr[:2000])
            raise RuntimeError('%s failed' % what)
        return r.stdout

    # -------------------------------------------- write the DWI MRtrix reads
    # b in s/mm^2, which is what MRtrix expects; this package carries ms/um^2
    # everywhere else. Voxel order is column-major over GRID_ALL, so voxel v of
    # S_noisy is voxel v of the image and Step 7 indexes them the same way.
    f_dwi  = os.path.join(MDIR, TAG + '_dwi')
    f_mask = os.path.join(MDIR, TAG + '_mask')
    S.write_mrtrix(f_dwi, S_noisy.reshape(GRID_ALL + [Ndwi], order='F'),
                   grad=np.column_stack([bvecs, bvals * 1000]))
    S.write_mrtrix(f_mask, np.ones(GRID_ALL), datatype='UInt8')
    print('   wrote %s_dwi.mih  [%s x %d] with dw_scheme' % (TAG, GRID_ALL, Ndwi))

    # CHECK. mrinfo must agree with what we think we wrote. This is the one
    # place the locally-implemented image writer is checked against MRtrix.
    sz_mr = [int(float(x)) for x in run(['mrinfo', '-size', f_dwi + '.mih'], 'mrinfo').split()]
    CHECK('mrinfo reads back the size we wrote', sz_mr == GRID_ALL + [Ndwi],
          'got %s' % sz_mr)

    # ----------------------------------- what MRtrix thinks the shells are
    b_mr = np.array([float(x) for x in
                     run(['mrinfo', '-shell_bvalues', f_dwi + '.mih'], 'mrinfo').split()])
    n_mr = np.array([int(float(x)) for x in
                     run(['mrinfo', '-shell_sizes', f_dwi + '.mih'], 'mrinfo').split()])
    print('   MRtrix shells: b = %s s/mm^2' % np.round(b_mr, 2).tolist())
    print('                  n = %s volumes' % n_mr.tolist())
    CHECK('MRtrix and this notebook agree on the shells',
          len(b_mr) == len(b_shell) and list(n_mr) == list(n_shell) and bool(np.all(np.diff(b_mr) > 0)))
    print('        shell   notebook b       MRtrix b     difference')
    for i in range(len(b_mr)):
        print('        %4d   %11.5f   %12.5f   %11.2e'
              % (i, b_shell[i], b_mr[i] / 1000, b_shell[i] - b_mr[i] / 1000))

    # ------------------------------------------ the single-shell subset
    # dwi2fod csd is single-shell and runs on the top shell, which is where SSST
    # CSD is normally run. dwiextract gets MRtrix's own shell b value, not the
    # nominal 3000, so it cannot miss a jittered volume.
    f_b3 = os.path.join(MDIR, TAG + '_b3')
    run(['dwiextract', f_dwi + '.mih', '-shells', '0,%g' % b_mr[-1],
         f_b3 + '.mih', '-force', '-quiet'], 'dwiextract')
    print('   top-shell subset for SSST-CSD: %s volumes'
          % run(['mrinfo', '-shell_sizes', f_b3 + '.mih'], 'mrinfo').strip())

    # --------------------------------- responses, one set per Lmax
    # Evaluated at MRtrix's own shell b values, and dispersion matched or not
    # according to RESPONSE_MODE.
    pl_full = S.watson_zonal_pl(KAPPA, LMAX_GT, dirs_q=dq, CS_phase=CS_PHASE)
    f_resp = {}
    for iL, Lf in enumerate(LMAX_LIST):
        pl_L = pl_full[:Lf // 2 + 1] if RESPONSE_MODE == 'dispersed' else None
        r_wm  = S.response_zh(K, b_mr / 1000, Lf, D_FW, pl=pl_L)
        r_gm  = (np.exp(-(b_mr / 1000) * D_GM) * np.sqrt(4 * np.pi))[:, None]
        r_csf = (np.exp(-(b_mr / 1000) * D_FW) * np.sqrt(4 * np.pi))[:, None]
        R = {'wm':    os.path.join(MDIR, '%s_resp_wm_lmax%d.txt' % (TAG, Lf)),
             'gm':    os.path.join(MDIR, '%s_resp_gm.txt' % TAG),
             'csf':   os.path.join(MDIR, '%s_resp_csf.txt' % TAG),
             'wm_b3': os.path.join(MDIR, '%s_resp_wm_b3_lmax%d.txt' % (TAG, Lf))}
        S.write_response(R['wm'], r_wm)
        S.write_response(R['gm'], r_gm)
        S.write_response(R['csf'], r_csf)
        S.write_response(R['wm_b3'], r_wm[-1:])       # single shell, one row
        f_resp[iL] = R
        e_rt = float(np.abs(S.read_response(R['wm']) - r_wm).max())
        CHECK('Lmax %d response survives the disk round trip' % Lf, e_rt < 1e-7,
              'max|err| = %.2e' % e_rt)

In [ ]:
if RUN_MRTRIX:
    SH_CSD, SH_MSMT = {}, {}
    tissue_share = np.full((len(LMAX_LIST), 3), np.nan)

    for iL, Lf in enumerate(LMAX_LIST):
        R  = f_resp[iL]
        nc = S.n_coef(Lf)
        f_msmt = os.path.join(MDIR, '%s_msmtfod_lmax%d' % (TAG, Lf))
        f_mgm  = os.path.join(MDIR, '%s_msmtgm_lmax%d' % (TAG, Lf))
        f_mcsf = os.path.join(MDIR, '%s_msmtcsf_lmax%d' % (TAG, Lf))
        f_csd  = os.path.join(MDIR, '%s_csdfod_lmax%d' % (TAG, Lf))

        t0 = time.time()
        run(['dwi2fod', 'msmt_csd', f_dwi + '.mih',
             R['wm'], f_msmt + '.mih', R['gm'], f_mgm + '.mih', R['csf'], f_mcsf + '.mih',
             '-mask', f_mask + '.mih', '-lmax', '%d,0,0' % Lf, '-force', '-quiet'],
            'dwi2fod msmt_csd')
        t_msmt = time.time() - t0

        t0 = time.time()
        run(['dwi2fod', 'csd', f_b3 + '.mih', R['wm_b3'], f_csd + '.mih',
             '-mask', f_mask + '.mih', '-lmax', str(Lf), '-force', '-quiet'],
            'dwi2fod csd')
        t_csd = time.time() - t0

        Vm = S.read_mrtrix(f_msmt + '.mih'); SH_MSMT[iL] = Vm.reshape(NVOX, Vm.shape[-1], order='F')
        Vc = S.read_mrtrix(f_csd + '.mih');  SH_CSD[iL]  = Vc.reshape(NVOX, Vc.shape[-1], order='F')
        Vg = S.read_mrtrix(f_mgm + '.mih');  Vf = S.read_mrtrix(f_mcsf + '.mih')

        print('   Lmax %d: msmt_csd %.1f s, csd %.1f s, %d coefficients each'
              % (Lf, t_msmt, t_csd, Vm.shape[-1]))

        # CHECK. The right number of coefficients, and nothing non-finite: a NaN
        # in an SH volume breaks downstream tractography silently.
        CHECK('Lmax %d: %d coefficients, all finite' % (Lf, nc),
              Vm.shape[-1] == nc and Vc.shape[-1] == nc
              and bool(np.isfinite(SH_MSMT[iL]).all()) and bool(np.isfinite(SH_CSD[iL]).all()))

        c0 = np.array([SH_MSMT[iL][:, 0].mean(), Vg.mean(), Vf.mean()])
        tissue_share[iL] = c0 / c0.sum()

    print('   MSMT-CSD tissue share (mean l=0 coefficient, normalised):')
    print('        Lmax        WM        GM       CSF')
    for iL, Lf in enumerate(LMAX_LIST):
        print('        %4d  %8.4f  %8.4f  %8.4f' % (Lf, *tissue_share[iL]))
    print('        Every simulated voxel is pure white matter with fw = %.2f, so anything'
          % K_WM[4])
    print('        outside the WM column is leakage. Section 6.2 of the Monte Carlo')
    print('        report found MSMT\'s CSF fraction is NOT a usable free-water estimate.')

    ARMS.append({'name': 'SSST-CSD', 'sh': SH_CSD})
    ARMS.append({'name': 'MSMT-CSD', 'sh': SH_MSMT})

NARM = len(ARMS)
print('\n   %d arm(s) to score: %s\n' % (NARM, ', '.join(a['name'] for a in ARMS)))

## Step 7 — peaks, angular error and spurious peaks

The only questions a tractography algorithm asks of an fODF are "how many
fibres, and pointing where", so those are what get scored.

Peaks are found by evaluating the fODF on a dense direction set and keeping
every direction not smaller than any neighbour within `PEAK_NBR` degrees, then
keeping those whose **anisotropic** amplitude is at least `PEAK_REL` of the
largest.

**The isotropic part is each voxel's own `l = 0` term, not the constant
`1/(4π)`.** That constant is the isotropic part only for an fODF with
`p_00 = 1` in every voxel, which is true of SMI's and false of an MRtrix FOD —
whose `l = 0` coefficient varies per voxel and carries apparent fibre density.
Subtracting the constant here would silently mis-threshold both arms.

Each Lmax is scored against **its own ceiling**: the ground truth truncated to
that same Lmax, with no noise and no deconvolution. That separates "the method
failed" from "this angular order cannot represent the answer". The ceiling is
not scored by a copy of the peak finder — the truth is prepended as the first
row of every block, so it goes through the same lines as the realisations.

**Four numbers per cell**, because a sweep is where they stop agreeing:
*correct count* (what tractography consumes), *mean angular error*, *its spread*,
and *spurious peaks per voxel*. A method can hold its angular error while it
starts inventing peaks, or lose the peak entirely and report a confident wrong
direction; no single column catches both.

In [ ]:
PEAK_NBR = 12.0     # degrees, neighbourhood for the local-maximum test
PEAK_REL = 0.30     # keep peaks at >= 30% of the largest anisotropic amplitude
de   = S.uniform_sphere_dirs(NDIR_E)
ND   = de.shape[0]
cosN = np.cos(np.radians(PEAK_NBR))
NBIDX = [np.where(de @ de[j] > cosN)[0] for j in range(ND)]   # includes j itself

print('Step 7: peaks (%d directions, %g deg neighbourhood, %.0f%% threshold)'
      % (ND, PEAK_NBR, 100 * PEAK_REL))
print('   The grid spacing sets a floor on angular error of roughly %.1f deg.'
      % (np.degrees(np.sqrt(4 * np.pi / ND)) / 2))

def score_block(A, axes_true):
    """Peaks of every row of A [nrow x ND], already isotropic-subtracted.

    Returns (nfound, aerr): how many peaks each row has, and the angle from its
    LARGEST peak to the nearest true axis.
    """
    nrow = A.shape[0]
    Amax = np.empty_like(A)
    for j in range(ND):
        Amax[:, j] = A[:, NBIDX[j]].max(axis=1)
    ismax = (A > 0) & (A >= Amax)
    nfound = np.zeros(nrow, dtype=int)
    aerr = np.full(nrow, np.nan)
    for r in range(nrow):
        lm = np.where(ismax[r])[0]
        if lm.size == 0:
            continue
        a = A[r]
        lm = lm[a[lm] >= PEAK_REL * a[lm].max()]
        lm = lm[np.argsort(-a[lm])]
        P = de[lm]
        sel = np.ones(P.shape[0], dtype=bool)
        for i in range(P.shape[0]):
            if not sel[i]:
                continue
            dup = np.abs(P @ P[i]) > cosN
            dup[i] = False
            sel[dup] = False
        P = P[sel]
        nfound[r] = P.shape[0]
        aerr[r] = min(np.degrees(np.arccos(min(abs(P[0] @ ax), 1.0))) for ax in axes_true)
    return nfound, aerr

shape = (NARM, len(LMAX_LIST), NSNR, NCOND)
res_all  = np.zeros(shape)          # % recovering the true fibre count
bias_all = np.full(shape, np.nan)   # mean angular error, deg
sd_all   = np.full(shape, np.nan)   # its standard deviation
med_all  = np.full(shape, np.nan)   # median angular error
spur_all = np.zeros(shape)          # spurious peaks per voxel
ceil_n   = np.zeros((NARM, len(LMAX_LIST), NCOND), dtype=int)
ceil_err = np.full((NARM, len(LMAX_LIST), NCOND), np.nan)

for ia, arm in enumerate(ARMS):
    print('\n   === %s ===' % arm['name'])
    for iL, Lf in enumerate(LMAX_LIST):
        Ye = S.get_even_SH(de, Lf, CS_PHASE)
        nc = Ye.shape[1]
        print('   Lmax %d' % Lf)
        print('     condition    SNR   correct count   mean err   std err   median err   spurious')
        for ic in range(NCOND):
            ntrue = len(AXES_GT[ic])
            for k in range(NSNR):
                isnr = SNR_ORD[k]
                rows = np.where((cond_id == ic) & (snr_id == isnr))[0]
                # Row 0 is the ground truth truncated to THIS Lmax; the rest are
                # the realisations. Same code path for both, by construction.
                blk = np.vstack([sh_gt[ic, :nc], arm['sh'][iL][rows]])
                A = blk @ Ye.T - blk[:, [0]] / np.sqrt(4 * np.pi)
                nfound, aerr = score_block(A, AXES_GT[ic])

                ceil_n[ia, iL, ic] = nfound[0]
                ceil_err[ia, iL, ic] = aerr[0]
                nf, ae = nfound[1:], aerr[1:]
                fin = np.isfinite(ae)
                res_all[ia, iL, isnr, ic] = 100 * np.mean(nf == ntrue)
                spur_all[ia, iL, isnr, ic] = np.mean(np.maximum(nf - ntrue, 0))
                if fin.any():
                    bias_all[ia, iL, isnr, ic] = ae[fin].mean()
                    med_all[ia, iL, isnr, ic] = np.median(ae[fin])
                if fin.sum() > 1:
                    sd_all[ia, iL, isnr, ic] = ae[fin].std(ddof=1)
                print('     %9s  %5s   %11.1f%%   %8.2f  %8.2f     %8.2f   %8.3f'
                      % (COLHDR[ic], SNR_LABEL[isnr], res_all[ia, iL, isnr, ic],
                         bias_all[ia, iL, isnr, ic], sd_all[ia, iL, isnr, ic],
                         med_all[ia, iL, isnr, ic], spur_all[ia, iL, isnr, ic]))
            print('     %9s ceiling  %10d peaks   %8.2f     (truth at Lmax %d)'
                  % (COLHDR[ic], ceil_n[ia, iL, ic], ceil_err[ia, iL, ic], Lf))

In [ ]:
# CHECK. The isotropic subtraction. For an SMI-convention fODF the per-voxel
# l = 0 term IS the constant 1/(4*pi); for an MRtrix FOD it is not. Verify that
# the general form reproduces the constant on the one input where both are
# defined -- the ground truth -- so the change of form cannot have broken it.
iso_general  = sh_gt[0, 0] / np.sqrt(4 * np.pi)
iso_constant = 1.0 / (4 * np.pi)
e_iso = abs(iso_general - iso_constant)
# Not required to be exactly zero, and it is not guaranteed to be: sh_gt[0,0] is
# 1/sqrt(4pi), so the general form evaluates (1/sqrt(4pi))*(1/sqrt(4pi)) where
# the constant form evaluates 1/(4pi) -- one rounding step apart. It happens to
# come out bit identical with numpy's evaluation order and does NOT in the
# Octave notebook, which measures 5.55e-17 on the same identity. Hence a
# tolerance rather than equality.
CHECK('per-voxel isotropic term == 1/(4pi) for the SMI convention', e_iso < 1e-15,
      'max|err| = %.2e, one rounding step apart at worst' % e_iso)

# CHECK. An MRtrix FOD is genuinely NOT on the SMI scale, which is why the
# constant would have been wrong. Measure the spread rather than assert it.
if NARM:
    l0 = ARMS[0]['sh'][0][:, 0]
    print('   %s l=0 coefficient: min %.4f, max %.4f (SMI convention would be %.4f)'
          % (ARMS[0]['name'], l0.min(), l0.max(), 1 / np.sqrt(4 * np.pi)))
    CHECK('MRtrix FOD is unnormalised, so the constant would have been wrong',
          float(l0.std()) > 0, 'std = %.4f across voxels' % float(l0.std()))

# CHECK. The noise-free arm must be deterministic. dwi2fod works one voxel at a
# time, so unlike SMI.fit these arms ARE bit identical voxel to voxel -- tested
# on the SH coefficients directly rather than through std() of a derived
# quantity, which is not a determinism test at all (std computes sum(x)/n first,
# and for n identical values that division need not return the value exactly).
i_inf = next((i for i in range(NSNR) if np.isinf(SNR_LIST[i])), None)
if i_inf is not None and NARM:
    for ia, arm in enumerate(ARMS):
        worst = 0.0
        for iL in range(len(LMAX_LIST)):
            for ic in range(NCOND):
                rows = np.where((cond_id == ic) & (snr_id == i_inf))[0]
                blk = arm['sh'][iL][rows]
                worst = max(worst, float(np.abs(blk - blk[0]).max()))
        CHECK('%s noise-free block is bit identical' % arm['name'], worst == 0.0,
              'max|row - row0| = %.2e' % worst)

# CHECK. More noise must not help. Compared at the extremes rather than pairwise
# so Monte Carlo error between adjacent SNRs cannot trip it.
if NSNR > 1:
    lo, hi = SNR_ORD[0], SNR_ORD[-1]
    ok_mono = True
    for ia in range(NARM):
        for iL in range(len(LMAX_LIST)):
            for ic in range(NCOND):
                a, b = bias_all[ia, iL, hi, ic], bias_all[ia, iL, lo, ic]
                if np.isfinite(a) and np.isfinite(b) and a > b:
                    ok_mono = False
    CHECK('more noise does not reduce the angular error', ok_mono,
          'SNR %s vs %s' % (SNR_LABEL[hi], SNR_LABEL[lo]))

### Three results in the output that are findings, not failures

Each looks alarming on first reading, so each is called out here rather than
left to be rediscovered.

**1. MSMT-CSD's noise-free angular error is far above the ceiling, and gets
worse with a blunter response.** Measured on the smoke configuration, 60°,
Lmax 6, `SNR = inf` — no noise at all:

| response given to the arms | MSMT peak errors | SSST peak errors |
|---|---|---|
| dispersion matched (`kappa = 16`) | **7.80°**, 3.54° | 1.27°, 1.72° |
| delta (exact kernel) | 2.82°, 3.54° | 1.27°, 1.72° |

SSST-CSD sits exactly on the ceiling (1.27°) either way. MSMT-CSD finds both
lobes in both cases — the peak *count* is correct — but its primary lobe is
displaced, and the displacement roughly triples when the response is blunted.

This is not a bug in the setup and it reproduces the published result from a
different direction: `Reports/REPORT_SMI_deconvolution_MonteCarlo.md` section
6.4 found MSMT-CSD's 60° error is **response-limited** (6.85° with an estimated
response, falling to 2.36° with the exact one). A dispersion-matched response is
deliberately blunter than the exact kernel, so MSMT degrading here is the same
effect measured at a third point on that axis.

**2. The angular error uses the largest peak only, and a symmetric crossing is a
near-tie.** MSMT's two lobes above differ by 7% in amplitude. Which one is
called "primary" can flip on the last few ulps, and the reported error then
jumps between the two lobes' errors without the reconstruction having changed.
Read the error alongside the peak count and the spurious count, never alone.
This is inherited deliberately: it is exactly what `smi_manuscript_60deg.m`
scores, and changing it here would make the arms incomparable.

**3. The band-limited ground truth is negative over roughly 40% of the sphere**
(Step 2). Truncating a Watson mixture rings and the rings cross zero, so the
non-negativity constraint CSD imposes is a regularizer, not a statement of fact.

## Step 8 — export for the `.m` file, and an independent peak finder

Everything above is scored by this notebook. That is fine for reading, but it is
not independent: the same code that read the fODF found its peaks. So two things
happen here.

**The independent check.** Every arm's fODFs are written as MRtrix SH images and
`sh2peaks` — which has never seen this package — is run on them. Both finders are
scored **against the true axes**, and the two errors compared. That is
deliberate: comparing the two finders' primary peaks *to each other* is the wrong
quantity, because a symmetric crossing has two lobes of near-equal amplitude and
which one is called "primary" is a tie broken by the last few ulps. Where the two
finders break it differently the peak-to-peak angle reads the crossing angle
itself and looks like a catastrophe while both finders are correct. What survives
as a real failure is **both columns large together** — and that is exactly how a
basis error presents: at `CS_phase = 1` every fODF is rotated 180° about z and
both columns would read ~71°.

**The export.** Everything the `.m` file needs to draw the manuscript figures:

| file | what |
|---|---|
| `csd_sh_<arm>_lmax<L>.bin` | `[NVOX x ncoef]` SH coefficients, one per arm per Lmax |
| `csd_scores.bin` | the five scored arrays, `[NARM x NLMAX x NSNR x NCOND x 5]` |
| `csd_signal.bin` | `[NVOX x Ndwi]` the noisy signal these arms were given |
| `csd_truth_sh.bin` | `[NCOND x ncoef]` the ground truth at `LMAX_GT` |
| `csd_manifest.txt` | the configuration, the arm and axis order, the voxel key |

`csd_signal.bin` is the one that matters most: it is what lets the `.m` file fit
SMI to **exactly these realisations** instead of its own, so the arms can be
compared without carrying the Monte Carlo error of two independent noise draws.

In [ ]:
os.makedirs(EDIR, exist_ok=True)
DATA = os.path.join(PKGDIR, 'data')
print('Step 8: export -> %s' % DATA)

# ---------------------------------------------------- arrays for the .m file
S.bin_save('csd_signal', S_noisy)
S.bin_save('csd_truth_sh', sh_gt)
S.bin_save('csd_bvals', bvals[:, None])
S.bin_save('csd_bvecs', bvecs)
for ia, arm in enumerate(ARMS):
    for iL, Lf in enumerate(LMAX_LIST):
        S.bin_save('csd_sh_%s_lmax%d' % (arm['name'].replace('-', ''), Lf), arm['sh'][iL])

# The five scored quantities in one array, in a fixed order the .m file reads
# back by index. Stacked rather than written as five files so they cannot get
# out of step with one another.
SCORES = np.stack([res_all, bias_all, sd_all, med_all, spur_all], axis=-1)
S.bin_save('csd_scores', SCORES)
S.bin_save('csd_ceiling', np.stack([ceil_n.astype(float), ceil_err], axis=-1))

# --------------------------------------------------------------- the manifest
man = os.path.join(DATA, 'csd_manifest.txt')
with open(man, 'w') as fh:
    fh.write('%% CSD Python arm, written by notebooks/csd_manuscript_60deg.ipynb\n')
    fh.write('%% Read with binio.m; every array is float64, column major.\n')
    fh.write('preset %s\n' % PRESET)
    fh.write('kernel %s\n' % ' '.join('%.10g' % v for v in K_WM))
    fh.write('D_FW %.10g\nD_GM %.10g\nkappa %.10g\n' % (D_FW, D_GM, KAPPA))
    fh.write('response_mode %s\n' % RESPONSE_MODE)
    fh.write('protocol %s\nb0_snap %.10g\n' % (PROTOCOL, B0_SNAP))
    fh.write('cs_phase %d\nlmax_gt %d\n' % (CS_PHASE, LMAX_GT))
    fh.write('smoke_test %d\nnrep %d\nseed %d\n' % (int(SMOKE_TEST), NREP, SEED))
    fh.write('angles %s\n' % ' '.join('%g' % a for a in ANGLES))
    fh.write('snr_list %s\n' % ' '.join(SNR_LABEL))
    fh.write('lmax_list %s\n' % ' '.join('%d' % v for v in LMAX_LIST))
    fh.write('arms %s\n' % ' '.join(a['name'] for a in ARMS))
    fh.write('score_order pct_correct mean_err std_err median_err spurious\n')
    fh.write('grid_all %s\n' % ' '.join('%d' % v for v in GRID_ALL))
    fh.write('nvox %d\nnvox_per_snr %d\nndwi %d\n' % (NVOX, NVOX_SNR, Ndwi))
    fh.write('axis1 %s\n' % ' '.join('%.12f' % v for v in AXIS1))
    for ic in range(NCOND):
        for j, ax in enumerate(AXES_GT[ic]):
            fh.write('axis_cond%d_%d %s\n' % (ic, j, ' '.join('%.12f' % v for v in ax)))
    fh.write('%% voxel key: voxel order is column major over grid_all,\n')
    fh.write('%% %d contiguous blocks of %d voxels, one per SNR, in the order %s.\n'
             % (NSNR, NVOX_SNR, ', '.join(SNR_LABEL)))
print('   csd_manifest.txt, csd_signal, csd_scores, csd_ceiling, csd_truth_sh')
print('   plus %d SH arrays (%d arms x %d Lmax)' % (NARM * len(LMAX_LIST), NARM, len(LMAX_LIST)))

# ------------------------------- MRtrix SH images, and the independent finder
if RUN_MRTRIX and RUN_SH2PEAKS and NARM:
    print('\n   sh2peaks cross-check (an independent peak finder, scored against the truth):')
    print('        arm         Lmax   SNR    this notebook    sh2peaks     agree')
    for ia, arm in enumerate(ARMS):
        nm = arm['name'].replace('-', '')
        for iL, Lf in enumerate(LMAX_LIST):
            fn = os.path.join(EDIR, '%s_%s_lmax%d' % (TAG, nm, Lf))
            S.write_mrtrix(fn, arm['sh'][iL].reshape(GRID_ALL + [arm['sh'][iL].shape[1]], order='F'))
            pk = fn + '_peaks'
            r = subprocess.run(['sh2peaks', fn + '.mih', pk + '.mih', '-num', '4',
                                '-force', '-quiet'], capture_output=True, text=True)
            if r.returncode != 0:
                print('        sh2peaks failed: %s' % r.stderr.strip()[:200])
                continue
            P = S.read_mrtrix(pk + '.mih').reshape(NVOX, -1, order='F')
            for k in range(NSNR):
                isnr = SNR_ORD[k]
                for ic in range(NCOND):
                    rows = np.where((cond_id == ic) & (snr_id == isnr))[0]
                    v = P[rows, 0:3]
                    n = np.linalg.norm(v, axis=1)
                    good = np.isfinite(n) & (n > 0)
                    if not good.any():
                        continue
                    u = v[good] / n[good][:, None]
                    e_sh2 = np.array([min(np.degrees(np.arccos(min(abs(ui @ ax), 1.0)))
                                          for ax in AXES_GT[ic]) for ui in u]).mean()
                    e_nb = bias_all[ia, iL, isnr, ic]
                    print('        %-10s  %4d  %5s   %10.2f deg  %10.2f deg   %s'
                          % (arm['name'], Lf, SNR_LABEL[isnr], e_nb, e_sh2,
                             VERDICT[abs(e_nb - e_sh2) < 5.0]))
    print('        Both columns are scored against the TRUE axes, so a lobe-swap')
    print('        between the two finders cannot show up as disagreement. Both')
    print('        columns large together is the real failure mode.')

## Figures

A **preview**, not the manuscript figures — those are drawn by
`smi_manuscript_60deg.m` from the arrays Step 8 exported, where they sit next to
the SMI arm. Three panels here, enough to see that the export is not nonsense
before it leaves Python.

| figure | what |
|---|---|
| 1 | the response CSD is given: delta against dispersion-matched, per shell. **The fibre axis points up**, so the waist is the signal's perpendicular maximum |
| 2 | the ground truth fODF and each arm's reconstruction, as amplitude profiles |
| 3 | mean angular error, spread and spurious peaks against SNR, one line per arm |

Set `MAKE_FIGURES = False` to skip them. Each is wrapped so a failure prints
rather than taking the run down — the printed tables above are the deliverable
either way, and every figure is drawn from the same arrays.

In [ ]:
MAKE_FIGURES = True
if MAKE_FIGURES:
    # Inline when there is a Jupyter kernel to draw into, Agg when this is being
    # run headless (nbconvert without a display, or as a plain script). Both
    # paths still write the PNGs, so the files exist either way.
    try:
        get_ipython().run_line_magic('matplotlib', 'inline')
    except Exception:
        import matplotlib
        matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    FIGDIR = os.path.join(EDIR, 'figures')
    os.makedirs(FIGDIR, exist_ok=True)
    print('figures -> %s' % FIGDIR)

In [ ]:
# ---- Figure 1: the response, delta vs dispersion matched, per shell
if MAKE_FIGURES:
    try:
        th = np.linspace(0, np.pi, 361)
        nsh = len(b_shell)
        fig, axs = plt.subplots(1, nsh, figsize=(3.1 * nsh, 3.3), subplot_kw={'polar': True})
        axs = np.atleast_1d(axs)
        for k in range(nsh):
            for R_, lab, st in [(R_delta, 'delta', '--'), (R_disp, 'dispersion matched', '-')]:
                pr = S.zh_profile(R_[k], th)
                axs[k].plot(th, np.abs(pr), st, lw=1.6, label=lab)
                axs[k].plot(-th, np.abs(pr), st, lw=1.6, color=axs[k].lines[-1].get_color())
            axs[k].set_title('b = %.2f' % b_shell[k], fontsize=10)
            # theta is measured FROM THE FIBRE AXIS, so put theta = 0 at the top:
            # matplotlib's polar default puts it at east, which draws the fibre
            # horizontally and the signal's perpendicular maximum vertically --
            # correct, and misread by everyone who has seen a response glyph.
            axs[k].set_theta_zero_location('N')
            axs[k].set_xticklabels([]); axs[k].set_yticklabels([])
        axs[0].legend(loc='upper left', bbox_to_anchor=(-0.35, 1.25), fontsize=8, frameon=False)
        fig.suptitle('Figure 1  the single-fibre response, per shell (RESPONSE_MODE = %r)'
                     % RESPONSE_MODE, fontsize=11)
        fig.tight_layout()
        fig.savefig(os.path.join(FIGDIR, 'fig1_response.png'), dpi=120)
        plt.show()
    except Exception as exc:
        print('** FIGURE 1 FAILED ** %s' % exc)

In [ ]:
# ---- Figure 2: truth and each arm's reconstruction, as amplitude profiles in
# the plane of the two fibres. A profile rather than a glyph because the point
# here is only "does the export look like a crossing", and a 2-D line is
# unambiguous where a flat-shaded 3-D glyph is not.
if MAKE_FIGURES and NARM:
    try:
        ic = 0
        a1, a2 = AXES_GT[ic][0], AXES_GT[ic][-1]
        e1 = a1 / np.linalg.norm(a1)
        e2 = a2 - (a2 @ e1) * e1
        e2 = e2 / np.linalg.norm(e2)
        t = np.linspace(0, 2 * np.pi, 721)
        ring = np.cos(t)[:, None] * e1 + np.sin(t)[:, None] * e2
        iL = 0; Lf = LMAX_LIST[iL]
        Yr_gt = S.get_even_SH(ring, LMAX_GT, CS_PHASE)
        Yr    = S.get_even_SH(ring, Lf, CS_PHASE)
        ncols = 1 + NARM
        fig, axs = plt.subplots(1, ncols, figsize=(3.2 * ncols, 3.4),
                                subplot_kw={'polar': True})
        axs = np.atleast_1d(axs)
        pr = Yr_gt @ sh_gt[ic] - sh_gt[ic, 0] / np.sqrt(4 * np.pi)
        axs[0].plot(t, np.abs(pr), lw=1.6)
        axs[0].set_title('truth (Lmax %d)' % LMAX_GT, fontsize=10)
        i_show = SNR_ORD[0]
        for ia, arm in enumerate(ARMS):
            rows = np.where((cond_id == ic) & (snr_id == i_show))[0]
            m = arm['sh'][iL][rows].mean(axis=0)
            pr = Yr @ m[:Yr.shape[1]] - m[0] / np.sqrt(4 * np.pi)
            axs[1 + ia].plot(t, np.abs(pr), lw=1.6, color='C%d' % (ia + 1))
            axs[1 + ia].set_title('%s, SNR %s' % (arm['name'], SNR_LABEL[i_show]), fontsize=10)
        # The true fibre axes, as RADIAL lines. In polar coordinates a radius is
        # plot([theta, theta], [0, rmax]); plotting two thetas at one radius
        # draws an arc instead, which is what a first attempt here did and it
        # showed nothing useful.
        for ax in axs:
            ax.set_theta_zero_location('N')     # the first fibre points up
            ax.set_xticklabels([]); ax.set_yticklabels([])
            rmax = ax.get_ylim()[1]
            for ang in (0.0, np.radians(ANGLES[ic])):
                for a in (ang, ang + np.pi):
                    ax.plot([a, a], [0, rmax], '-', color='0.55', lw=0.9,
                            alpha=0.8, zorder=0)
        fig.suptitle('Figure 2  in the plane of the two fibres, Lmax %d '
                     '(grey lines = true axes)' % Lf, fontsize=11)
        fig.tight_layout()
        fig.savefig(os.path.join(FIGDIR, 'fig2_fodf.png'), dpi=120)
        plt.show()
    except Exception as exc:
        print('** FIGURE 2 FAILED ** %s' % exc)

In [ ]:
# ---- Figure 3: the summary curves. One row per metric, one line per arm.
if MAKE_FIGURES and NARM and NSNR > 1:
    try:
        finite = [i for i in SNR_ORD if np.isfinite(SNR_LIST[i])]
        xs = [SNR_LIST[i] for i in finite]
        metrics = [(bias_all, 'mean angular error (deg)'),
                   (sd_all,   'std of angular error (deg)'),
                   (spur_all, 'spurious peaks per voxel')]
        fig, axs = plt.subplots(1, 3, figsize=(12, 3.6))
        for k, (arr, lab) in enumerate(metrics):
            for ia, arm in enumerate(ARMS):
                for iL, Lf in enumerate(LMAX_LIST):
                    ys = [arr[ia, iL, i, 0] for i in finite]
                    axs[k].plot(xs, ys, 'o-', lw=1.4, ms=4,
                                label='%s Lmax %d' % (arm['name'], Lf))
            axs[k].set_xscale('log'); axs[k].set_xlabel('SNR')
            axs[k].set_ylabel(lab); axs[k].grid(alpha=0.3)
        axs[0].legend(fontsize=8, frameon=False)
        fig.suptitle('Figure 3  %d deg crossing, %s kernel, %s response'
                     % (ANGLES[0], PRESET, RESPONSE_MODE), fontsize=11)
        fig.tight_layout()
        fig.savefig(os.path.join(FIGDIR, 'fig3_summary.png'), dpi=120)
        plt.show()
    except Exception as exc:
        print('** FIGURE 3 FAILED ** %s' % exc)

In [ ]:
# ======================= the audit line =======================
nfail = sum(1 for _, ok in CHECKS if not ok)
print('\n' + '=' * 68)
print('%d of %d CHECKs pass.' % (len(CHECKS) - nfail, len(CHECKS)))
if nfail:
    print('FAILED:')
    for label, ok in CHECKS:
        if not ok:
            print('   %s' % label)
else:
    print('Every check passed. Results are in %s, ready for the .m file.'
          % os.path.join(PKGDIR, 'data'))
print('=' * 68)

## Step 9 — from here to a full campaign, and into the `.m` file

**Set `SMOKE_TEST = False`.** That is the whole difference between this notebook
and the manuscript configuration: `NREP = 1000`, seven SNR levels and three
angular orders. The MRtrix arms scale with voxel count rather than with fit
count, so this notebook stays in minutes where the SMI arm takes hours.

### Handing the results to the `.m` file

From MATLAB or Octave, in `deconv_comparison/`:

```matlab
run('oct_path.m');
C = read_csd_export();       % everything this notebook wrote
```

`read_csd_export.m` returns a struct with the arm names, the scored arrays, the
SH coefficients, the noisy signal and the full configuration, and **re-scores the
SH coefficients with the .m file's own peak finder** to confirm it reproduces the
numbers printed above. Run `test_csd_roundtrip.m` for that comparison on its own.

The intended use in `smi_manuscript_60deg.m` is:

1. fit SMI to `C.signal` rather than to a fresh noise draw, so all three arms sit
   on **the same realisations**;
2. append `C.arms` to its own arm list;
3. draw Figures 4–6 with the extra columns and curves.

### What is deliberately not here

- **No SMI arm.** It is the `.m` file's job, and it takes hours where this takes
  minutes. Splitting them is the reason the signal is exported.
- **No edema kernel.** This is the healthy baseline. The edema kernel is a
  one-line change in the Configuration cell (`K_WM = [0.10 2.4 2.7 1.15 0.35]`),
  but the two-kernel comparison lives in the `.m` file.
- **No estimated response.** `RESPONSE_MODE` chooses between the exact delta
  response and the dispersion-matched one. Swapping in a `dwi2response` output is
  a `S.read_response(...)` call in Step 6 — but the responses committed in
  `mrtrix_responses/` were estimated on the *superseded* synthetic protocol, so
  they cannot be dropped in without regenerating them on this acquisition.
- **No real data.** Every number in this notebook, and in every report in this
  repository, is simulation.